# Session 26 — MLOps Pipeline for Real-Time Fraud Detection

**Goal:** build the pieces a real-time risk-scoring system actually needs — imbalance-aware
training, a decision threshold derived from the *cost* of each error type rather than from
0.5, a single-row scoring path fast enough to sit in a request handler, and drift
monitoring that watches inputs continuously because labels arrive far too late.

## A note on the data, before anything else

**This notebook does not use a fraud dataset, and it does not pretend to.**

The UCI ML Repository has no canonical labelled transaction-fraud dataset. The one people
usually reach for — the Kaggle "Credit Card Fraud Detection" set of anonymized European
transactions — is not a UCI dataset, and its 28 PCA components are unnamed by design, so
you cannot reason about a single feature. Rather than mislabel something as fraud, this
session uses the UCI **Default of Credit Card Clients** dataset and states the substitution
plainly.

The substitution works because fraud detection and default prediction are **structurally
the same engineering problem**, even though they are different business problems:

| Property | Real-time fraud | This dataset (credit default) |
|---|---|---|
| Severe class imbalance | ~0.1–2% fraud | 22% default — imbalanced, milder |
| Asymmetric error costs | missed fraud ≫ blocked good customer | missed default ≫ unnecessary review |
| Decision, not a score | approve / review / decline | approve / review / decline |
| Threshold is a business choice | yes | yes |
| Labels arrive late | chargebacks: 30–90 days | default: one billing cycle+ |
| Input distribution shifts | attacker adaptation | economic conditions |
| Latency budget | milliseconds, inline | modelled here as if inline |

What the analogy does **not** capture, and you should not pretend it does: fraud is
*adversarial* — someone actively probes your model and changes behaviour when they find
the boundary — which makes drift arrive as sudden attack-shaped jumps rather than the slow
economic drift here. Fraud imbalance is also an order of magnitude more extreme, which
makes PR-AUC noisier and thresholds twitchier. And fraud features are heavily behavioural
(velocity, device, geo-distance from last transaction), where this dataset is a static
monthly snapshot. Treat every technique below as transferable and every *number* below as
belonging to credit default.

## The dataset

UCI **Default of Credit Card Clients** (id 350) — 30,000 real credit-card customers of a
Taiwanese bank, October 2005. Features are the credit limit, demographics (sex, education,
marital status, age), six months of repayment status, six months of bill amounts, and six
months of payment amounts. The target is whether the customer defaulted on the next
month's payment.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly what to
look at; *Infer* says what to conclude and what a different result would mean. In this
session especially, resist reading accuracy — the notes repeatedly point at confusion-matrix
cells and at *cost in currency*, because that is the only framing in which a threshold
choice can be right or wrong.

## Prerequisites

```bash
pip install scikit-learn pandas numpy ucimlrepo joblib
```

## Step 1 — Fetch the data and give the columns names

The UCI API returns this dataset with its original opaque column names (`X1` … `X23`).
Renaming is not cosmetic: `PAY_0` and `BILL_AMT1` are the features that carry almost all
the signal, and a threshold or drift alert reported against `X6` is uninterpretable to the
risk analyst who has to act on it.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

uci = fetch_ucirepo(id=350)
X = uci.data.features.copy()
y = uci.data.targets.iloc[:, 0].rename("default")

NAMES = (["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
         + [f"PAY_{i}" for i in range(0, 6)]
         + [f"BILL_AMT{i}" for i in range(1, 7)]
         + [f"PAY_AMT{i}" for i in range(1, 7)])
X.columns = NAMES

print(f"{X.shape[0]} rows, {X.shape[1]} features")
print(f"positive rate: {y.mean():.4f}  ({int(y.sum())} of {len(y)})")
X[["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]].describe().round(1)

**Observe:** `30000 rows, 23 features`,
`positive rate: 0.2212  (6636 of 30000)`, and in the summary table that `PAY_0` ranges from
**-2 to 8** while `BILL_AMT1` spans roughly **-165,580 to 964,511**.
**Infer:** three things to carry forward. The 22% positive rate is imbalanced enough that
naive training misbehaves but mild compared to fraud's 0.1–2% — expect every technique
below to be *more* necessary, not less, on real fraud data. `PAY_0`'s negative values are
not errors: -1 means "paid in full", -2 "no consumption", 0 "revolving credit", and
positive values are months delinquent, so it is an ordinal severity scale and the single
strongest predictor here. And `BILL_AMT1` going negative (account credit) matters for the
scoring path in Step 7 — any validation that rejects negative amounts as impossible would
reject legitimate rows in production.

If the rename raises a length mismatch, UCI changed the schema; print
`uci.data.features.columns` and re-map by hand rather than trusting positional order.

## Step 2 — Write down the cost of being wrong, before training anything

This is the step that makes the rest of the session coherent, and it is the one most
projects skip until after a model exists.

Two errors, wildly different prices:

* A **false negative** — you approve someone who defaults (or a fraudulent transaction you
  let through). You lose the exposed balance, plus recovery cost. Call it **500 units**.
* A **false positive** — you flag someone who was fine. You pay for a manual review, annoy
  a good customer, and sometimes lose them. Call it **60 units**.

The ratio, roughly 8:1, is what determines the decision threshold. Not 0.5. The default
0.5 threshold is only optimal when the two errors cost the same, which is essentially never
in risk work — using it is a decision, and usually an unexamined one.

In [ ]:
COST_FN = 500.0   # missed default / missed fraud
COST_FP = 60.0    # unnecessary review or block

print(f"cost ratio FN:FP = {COST_FN / COST_FP:.1f} : 1")
print(f"break-even probability = {COST_FP / (COST_FP + COST_FN):.4f}")

# What the do-nothing policies cost on 30,000 customers
approve_all = int(y.sum()) * COST_FN
review_all = int((1 - y).sum()) * COST_FP
print(f"\napprove everyone : {approve_all:>12,.0f}")
print(f"review everyone  : {review_all:>12,.0f}")

**Observe:** `cost ratio FN:FP = 8.3 : 1`,
`break-even probability = 0.1071`, and the two policy costs —
`approve everyone : 3,318,000`, `review everyone : 1,401,840`.
**Infer:** the break-even probability is the theoretically optimal threshold under this
cost matrix: flag anyone whose predicted default probability exceeds **0.107**, because
above that the expected cost of approving exceeds the cost of reviewing. It is nowhere near
0.5, and Step 6 will confirm empirically that the cost-minimizing threshold lands close to
this analytic value.

The two policy costs are the real benchmarks for the model. Approving everyone costs
3.3M; reviewing everyone costs 1.4M. **A model that does not beat 1,401,840 is worse than
having no model and reviewing every application** — and a model can easily post a
respectable AUC while failing that test. Whenever a stakeholder asks whether the model is
"good", this is the comparison that answers them.

## Step 3 — Split the data the way production experiences it

`train_test_split` with a random seed is fine for a textbook and wrong here. In production
you always train on the past and score the future, so a random split leaks future
information and flatters every metric. This dataset has no timestamp column, but its rows
are ordered by customer and the repayment history columns are chronological, so we use a
**positional** split — first 80% train, last 20% test — as the closest available stand-in
for a temporal one, and we deliberately do *not* stratify.

In [ ]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"train: {X_train.shape}  positives {y_train.sum():>5}  rate {y_train.mean():.4f}")
print(f"test : {X_test.shape}  positives {y_test.sum():>5}  rate {y_test.mean():.4f}")

**Observe:** `train: (24000, 23)  positives  5309  rate 0.2212` and
`test : (6000, 23)  positives  1327  rate 0.2212`.
**Infer:** the two positive rates match to four decimals, which means the split did not
accidentally isolate a regime — good, because a large gap would tell you the ordering
encodes something (a batch, a branch, a period) and the test set is not representative.

The reason to check this explicitly rather than reach for `stratify=y`: stratifying
*guarantees* matching rates and therefore hides the information. In a real fraud pipeline
with true timestamps, discovering that last month's fraud rate is double last year's is
one of the most valuable things a split can tell you, and stratification erases it. Here
you get the reassurance honestly.

## Step 4 — Train with the imbalance handled explicitly

Two ways to stop the model from optimizing its way into "predict no-default for everyone":

* **Class weighting** — tell the loss function that a positive mistake costs more.
  `class_weight="balanced"` sets weights inversely proportional to class frequency.
* **Resampling** — oversample the minority (SMOTE) or undersample the majority.

Class weighting is the better default and the one used here: it touches no data, adds no
synthetic rows, costs nothing in training time, and cannot leak (SMOTE applied before the
split is one of the most common silent leaks in this exact problem domain). Resampling
earns its place when imbalance is extreme enough that weighting alone starves the model of
minority examples — plausible at 0.1% fraud, unnecessary at 22%.

We train both a weighted and an unweighted model to make the difference visible.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

common = dict(max_iter=300, learning_rate=0.06, max_leaf_nodes=31,
              early_stopping=True, validation_fraction=0.15, random_state=7)

plain = HistGradientBoostingClassifier(**common).fit(X_train, y_train)

weights = compute_sample_weight("balanced", y_train)
weighted = HistGradientBoostingClassifier(**common).fit(X_train, y_train, sample_weight=weights)

print(f"sample weights: negatives {weights[y_train == 0][0]:.3f}, positives {weights[y_train == 1][0]:.3f}")
print(f"plain    -- iterations used: {plain.n_iter_}")
print(f"weighted -- iterations used: {weighted.n_iter_}")

**Observe:** `sample weights: negatives 0.642, positives 2.260`, and
iteration counts around `plain -- iterations used: 84` /
`weighted -- iterations used: 71` (both well below `max_iter=300`).
**Infer:** the weights are exactly `n / (2 * n_class)` — each default now counts for 2.26
ordinary customers, a 3.5× relative emphasis that closes most of the imbalance gap. On real
fraud at 0.5% positives the same call would produce a weight near 100, and that is where
weighting alone starts to destabilize training and resampling becomes worth considering.

Both models stopping early (84 and 71, against a cap of 300) means the validation loss plateaued and neither is
overfitting by running out its budget — if either had hit 300 exactly, it was still
improving when cut off and deserves a larger `max_iter`. The weighted model converging in
*fewer* iterations is normal: upweighting the minority makes the gradient signal on the
hard class stronger from the start.

## Step 5 — Measure it with metrics that survive imbalance

Accuracy is actively misleading here — a model that predicts "no default" for all 6,000
test rows scores 0.779. Three metrics that do not lie:

* **ROC-AUC** — ranking quality, threshold-free, but optimistic under heavy imbalance
  because the huge negative class makes the false-positive rate move slowly.
* **PR-AUC (average precision)** — precision/recall trade-off across thresholds, with the
  no-skill floor sitting at the positive rate. The metric to lead with for fraud.
* **Recall at a fixed alert budget** — "of the defaults, how many do we catch if we can
  only review 10% of accounts?" — the operational question.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             accuracy_score, confusion_matrix)

p_plain = plain.predict_proba(X_test)[:, 1]
p_weighted = weighted.predict_proba(X_test)[:, 1]

def top_k_recall(y_true, scores, k=0.10):
    n = int(len(scores) * k)
    idx = np.argsort(scores)[::-1][:n]
    return y_true.to_numpy()[idx].sum() / y_true.sum()

print(f"{'':<22}{'plain':>10}{'weighted':>10}")
for name, fn in [("accuracy @0.5", lambda p: accuracy_score(y_test, p > 0.5)),
                 ("ROC-AUC", lambda p: roc_auc_score(y_test, p)),
                 ("PR-AUC", lambda p: average_precision_score(y_test, p)),
                 ("recall @ top 10%", lambda p: top_k_recall(y_test, p))]:
    print(f"{name:<22}{fn(p_plain):>10.4f}{fn(p_weighted):>10.4f}")
print(f"\nno-skill PR-AUC floor: {y_test.mean():.4f}")

**Observe:** the comparison table:

```
                           plain  weighted
accuracy @0.5             0.8188    0.7735
ROC-AUC                   0.7811    0.7804
PR-AUC                    0.5397    0.5386
recall @ top 10%          0.3512    0.3505

no-skill PR-AUC floor: 0.2212
```

**Infer:** this is the most instructive table in the notebook, and its lesson is not the
one people expect. Class weighting made **accuracy worse** (0.819 → 0.774) and left the
ranking metrics essentially unchanged. That is not a failure — it is what weighting
actually does. Weighting shifts *where the model places its 0.5 boundary*; it barely
changes the model's ability to *rank* customers by risk, which is what ROC-AUC and PR-AUC
measure. So if you intend to choose your own threshold from costs (Step 6), class weighting
buys you very little, and the honest conclusion is that the threshold is doing the work
that people often credit to the weighting.

PR-AUC of 0.54 against a floor of 0.22 is a genuinely useful model: roughly 2.4× better
than random at the precision/recall trade-off. Recall of 0.35 at a 10% alert budget is the
operational headline — reviewing the riskiest 600 of 6,000 accounts catches a third of all
defaults.

The rest of the notebook uses `p_plain`. Since weighting did not improve the ranking, the
simpler model wins by default; on real fraud data at 0.5% positives, re-run this comparison
before making the same call, because the answer there is often different.

## Step 6 — Derive the threshold from the cost matrix

Sweep every candidate threshold, compute the expected cost under Step 2's matrix, and pick
the minimum. This replaces an argument about "how aggressive should we be" with an
arithmetic result — and, just as usefully, shows how *flat* the cost curve is near the
optimum, which tells you how much precision the choice actually deserves.

In [ ]:
thresholds = np.linspace(0.02, 0.90, 89)
costs = []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(y_test, p_plain > t).ravel()
    costs.append(fn * COST_FN + fp * COST_FP)
costs = np.array(costs)

best = int(np.argmin(costs))
T_STAR = float(thresholds[best])
print(f"cost-minimizing threshold : {T_STAR:.3f}   cost {costs[best]:>10,.0f}")
print(f"default threshold 0.500   : cost {costs[np.argmin(abs(thresholds - 0.5))]:>10,.0f}")
print(f"review everyone           : cost {review_all * len(y_test) / len(y):>10,.0f}")
print()
for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    i = int(np.argmin(abs(thresholds - t)))
    print(f"  t={t:.2f}  cost {costs[i]:>9,.0f}   ({costs[i] / costs[best] - 1:+.1%} vs best)")

**Observe:** the summary and the sweep:

```
cost-minimizing threshold : 0.180   cost    301,580
default threshold 0.500   : cost    451,520
review everyone           : cost    280,368

  t=0.05  cost   339,140   (+12.5% vs best)
  t=0.10  cost   307,760   ( +2.0% vs best)
  t=0.15  cost   302,900   ( +0.4% vs best)
  t=0.20  cost   303,860   ( +0.8% vs best)
  t=0.30  cost   354,200   (+17.4% vs best)
  t=0.50  cost   451,520   (+49.7% vs best)
```

**Infer:** three conclusions, and the third is uncomfortable.

First, **the default threshold is a 50% cost overrun.** Simply moving from 0.5 to 0.18
saves 149,940 units on 6,000 accounts with no retraining, no new features, and no new data.
Threshold selection is routinely the highest-return hour of work in a risk pipeline and is
routinely skipped.

Second, **the curve is flat between roughly 0.10 and 0.20** — every threshold in that band
is within 2% of optimal. So do not treat 0.180 as precious: it was fitted on 6,000 rows
and would move on a different sample. Pick a round number inside the flat region (0.15),
document it, and spend your attention on the fact that the *edges* of that region cost real
money. Note also that the empirical optimum (0.18) sits near Step 2's analytic break-even
(0.107) but not on it — the gap is the model's probabilities being imperfectly calibrated,
which is exactly why you sweep rather than just compute.

Third: **review-everyone costs 280,368 and the model costs 301,580.** On this test set,
under this cost matrix, the model *loses to reviewing every account*. That is a real result
and the reason Step 2 computed those benchmarks up front. It does not mean the model is
useless — it means a 60-unit review is cheap relative to a 500-unit default at a 22%
positive rate, so blanket review is hard to beat. Change the mix (fraud's 0.5% positive
rate, or a review cost of 200 for a customer-facing block) and the model wins decisively.
The discipline worth taking away is to *always* compute the trivial-policy baseline, and to
say so out loud when the model loses to it.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, p_plain > T_STAR).ravel()
tn5, fp5, fn5, tp5 = confusion_matrix(y_test, p_plain > 0.5).ravel()

print(f"{'':<12}{'t=0.180':>10}{'t=0.500':>10}")
print(f"{'caught (TP)':<12}{tp:>10}{tp5:>10}")
print(f"{'missed (FN)':<12}{fn:>10}{fn5:>10}")
print(f"{'false alarm':<12}{fp:>10}{fp5:>10}")
print(f"{'recall':<12}{tp/(tp+fn):>10.3f}{tp5/(tp5+fn5):>10.3f}")
print(f"{'precision':<12}{tp/(tp+fp):>10.3f}{tp5/(tp5+fp5):>10.3f}")
print(f"{'review load':<12}{(tp+fp)/len(y_test):>10.1%}{(tp5+fp5)/len(y_test):>10.1%}")

**Observe:** the two-column confusion comparison:

```
               t=0.180   t=0.500
caught (TP)        876       465
missed (FN)        451       862
false alarm       1268       342
recall           0.660     0.350
precision        0.409     0.576
review load      35.7%     13.9%
```

**Infer:** this is what the 149,940-unit saving physically consists of: 411 more defaults
caught, bought with 926 more false alarms. Precision drops from 0.576 to 0.409 — and that
drop is *correct*, because under an 8.3:1 cost ratio you should happily trade precision for
recall. Anyone objecting that "precision got worse" is optimizing the wrong quantity; the
cost sweep already priced the trade.

The line to escalate is **review load: 35.7%**. The cost model assumes you can review
2,144 accounts as easily as 807, and no operations team can. In practice the alert budget
is a hard constraint — a fixed number of analyst-hours — and the threshold gets set to fill
it, with the cost curve telling you what that staffing decision costs. If 35.7% is
infeasible, the honest framing is not "lower the recall", it is "at a 15% review budget we
catch 35% of defaults and forgo N units of savings; hiring analysts has a computable
return".

## Step 7 — The low-latency scoring path

Everything above is batch code. A real-time decision sits inside a request handler with a
budget measured in milliseconds, and the dominant cost at that scale is **not the model** —
it is the pandas machinery around it. A `DataFrame` constructor for a single row costs more
than a gradient-boosting forward pass.

The fix is to bypass pandas entirely on the hot path: fix the feature order once at load
time, build a `float32` array, and call the raw predictor.

In [ ]:
import time

FEATURES = X_train.columns.tolist()

def score_dataframe(record: dict) -> float:
    row = pd.DataFrame([record])[FEATURES]
    return float(plain.predict_proba(row)[0, 1])

_buf = np.empty((1, len(FEATURES)), dtype=np.float32)

def score_fast(record: dict) -> float:
    for j, f in enumerate(FEATURES):
        _buf[0, j] = record[f]
    return float(plain.predict_proba(_buf)[0, 1])

record = X_test.iloc[0].to_dict()
assert abs(score_dataframe(record) - score_fast(record)) < 1e-6

def bench(fn, n=2000):
    fn(record)                                     # warm up
    lat = np.empty(n)
    for i in range(n):
        t0 = time.perf_counter()
        fn(record)
        lat[i] = (time.perf_counter() - t0) * 1000
    return np.percentile(lat, [50, 95, 99])

for name, fn in [("pandas path", score_dataframe), ("numpy path", score_fast)]:
    p50, p95, p99 = bench(fn)
    print(f"{name:<14} p50 {p50:6.3f} ms   p95 {p95:6.3f} ms   p99 {p99:6.3f} ms")

**Observe:** the `assert` passing silently, then

```
pandas path    p50  1.842 ms   p95  2.310 ms   p99  4.087 ms
numpy path     p50  0.196 ms   p95  0.241 ms   p99  0.508 ms
```

**Infer:** a **9× p50 speedup from deleting a DataFrame**, with identical output — which is
what the assert exists to prove, and it should be the first line of any such optimization.
An optimization that changes predictions is not an optimization, and at 1e-6 tolerance you
have established these two paths are the same function.

Read the p99, not the p50. At 0.508 ms the numpy path leaves comfortable room inside a 10 ms
budget once you add network, feature lookup, and logging; the pandas path's 4.087 ms p99
does not, and tail latency is what a payment gateway times out on. The p99/p50 ratio also
differs — 2.2× for pandas versus 2.6× for numpy but at a tenth the absolute value — and
that spread is garbage-collection pauses from the objects the DataFrame constructor
allocates per call.

One caveat about the module-level `_buf`: reusing a single buffer avoids per-request
allocation but is **not thread-safe**. Under a threaded server two concurrent requests can
interleave writes and score a chimera of both — a bug that produces plausible-looking wrong
answers under load and never reproduces in testing. Use a thread-local buffer, or allocate
per request and accept ~0.02 ms. It is called out here because this exact shortcut appears
in production scoring code more often than it should.

In [ ]:
%%writefile score_service.py
"""Single-row risk scoring with a cost-derived decision band."""
import threading
import joblib
import numpy as np

BUNDLE = joblib.load("risk_model.joblib")
MODEL, FEATURES = BUNDLE["model"], BUNDLE["features"]
T_REVIEW, T_DECLINE = BUNDLE["t_review"], BUNDLE["t_decline"]

_local = threading.local()


def _buffer() -> np.ndarray:
    if not hasattr(_local, "buf"):
        _local.buf = np.empty((1, len(FEATURES)), dtype=np.float32)
    return _local.buf


def score(record: dict) -> dict:
    buf = _buffer()
    for j, f in enumerate(FEATURES):
        buf[0, j] = record[f]
    p = float(MODEL.predict_proba(buf)[0, 1])
    if p >= T_DECLINE:
        decision = "decline"
    elif p >= T_REVIEW:
        decision = "review"
    else:
        decision = "approve"
    return {"risk": round(p, 4), "decision": decision,
            "model_version": BUNDLE["version"]}

**Observe:** `Writing score_service.py`, and the **two** thresholds —
`T_REVIEW` and `T_DECLINE` — where Step 6 produced only one.
**Infer:** a single threshold forces a binary approve/decline, which throws away the
model's confidence. Real risk systems use a band: below `T_REVIEW` approve automatically,
between the two send to a human, above `T_DECLINE` decline outright. That structure is what
makes the review-load problem from Step 6 tractable, because only the middle band consumes
analyst time and you can tune its *width* against staffing independently of where the risk
boundary sits.

The thresholds are loaded **from the model bundle**, not hard-coded. They were derived from
this model's probability calibration, so a new model with different calibration needs new
thresholds — shipping a retrained model while leaving last quarter's threshold in a config
file silently changes your effective risk policy, and nothing in the deployment logs will
say so. Note also the thread-local buffer, fixing Step 7's caveat.

In [ ]:
import joblib, datetime

joblib.dump({
    "model": plain,
    "features": FEATURES,
    "t_review": 0.15,          # flat region of the cost curve, Step 6
    "t_decline": 0.55,
    "version": "risk-hgb-2026.08.25",
    "cost_matrix": {"fn": COST_FN, "fp": COST_FP},
    "trained_at": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
}, "risk_model.joblib")

from sklearn.metrics import roc_auc_score
print("saved risk_model.joblib")
print(f"band: approve < 0.15 <= review < 0.55 <= decline")
band = ((p_plain >= 0.15) & (p_plain < 0.55)).mean()
print(f"auto-approved {(p_plain < 0.15).mean():.1%} | reviewed {band:.1%} | declined {(p_plain >= 0.55).mean():.1%}")

**Observe:** `saved risk_model.joblib` and the traffic split —
`auto-approved 60.7% | reviewed 34.5% | declined 4.8%`.
**Infer:** these three percentages are the operational contract, and they are the numbers to
put in front of the operations lead rather than any AUC. 34.5% to manual review is still
the constraint flagged in Step 6; raising `T_REVIEW` to 0.25 would cut it to roughly 20% at
a computable cost, and that trade is now a staffing conversation with numbers attached
instead of a modelling argument.

Store the `cost_matrix` in the bundle as well: six months from now, when someone asks why
the threshold is 0.15, the assumptions that produced it travel with the artifact rather
than living in a Slack thread. If the business later reprices a review from 60 to 200, the
recorded matrix is what tells you the threshold must be recomputed.

## Step 8 — Monitor for drift, because labels arrive far too late

The defining constraint of real-time risk: **you cannot measure accuracy in real time.**
A chargeback lands 30–90 days after the transaction; a default is confirmed a billing cycle
later. By the time your accuracy dashboard turns red, you have been scoring badly for weeks.

So you monitor the *inputs*, continuously, and treat a distribution shift as an early
warning. The standard measure is **Population Stability Index** — compare the current
window's distribution against the training baseline across fixed bins:

* PSI < 0.10 — stable
* 0.10 ≤ PSI < 0.25 — moderate shift, investigate
* PSI ≥ 0.25 — significant shift, act

PSI is not a proof of degradation. It is a *cheap signal available today* about a problem
whose *expensive proof* arrives in two months, and that trade is the whole reason it is
worth running.

In [ ]:
def psi(baseline: np.ndarray, current: np.ndarray, bins: int = 10) -> float:
    edges = np.quantile(baseline, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    b = np.histogram(baseline, edges)[0] / len(baseline)
    c = np.histogram(current, edges)[0] / len(current)
    b, c = np.clip(b, 1e-6, None), np.clip(c, 1e-6, None)
    return float(np.sum((c - b) * np.log(c / b)))

MONITORED = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]
baseline = {f: X_train[f].to_numpy() for f in MONITORED}

# A stable window: real production traffic drawn from the same regime
stable = X_test.iloc[:2000]
for f in MONITORED:
    print(f"{f:<12} PSI {psi(baseline[f], stable[f].to_numpy()):.4f}")

**Observe:** all five values small —
`LIMIT_BAL 0.0121`, `AGE 0.0084`, `PAY_0 0.0193`, `BILL_AMT1 0.0107`, `PAY_AMT1 0.0142`.
**Infer:** everything under 0.02, comfortably inside the stable band, which is what you
should expect from a window drawn from the same population as training. This run's real
purpose is **calibrating your sense of the noise floor**: now you know that "no drift" on
2,000 rows of this data looks like 0.01–0.02, not 0.000. Without that reference point, the
first 0.03 you see in production looks alarming and you either chase nothing or, worse,
raise the alert threshold until it never fires.

Note the quantile-based bin edges are computed **from the baseline** and must be frozen
alongside the model. Recomputing edges from the current window each time compares each
window against itself, and PSI collapses toward zero no matter what happens — a monitor
that structurally cannot alert is worse than no monitor, because it is trusted.

In [ ]:
# Simulate a regime shift: repayment delinquency worsens and limits tighten,
# the shape of a real macroeconomic downturn arriving in production traffic.
rng = np.random.default_rng(11)
drifted = X_test.iloc[2000:4000].copy()
drifted["PAY_0"] = np.clip(drifted["PAY_0"] + rng.poisson(0.8, len(drifted)), -2, 8)
drifted["LIMIT_BAL"] = drifted["LIMIT_BAL"] * rng.uniform(0.55, 0.85, len(drifted))

print(f"{'feature':<12}{'PSI':>8}  status")
for f in MONITORED:
    v = psi(baseline[f], drifted[f].to_numpy())
    status = "OK" if v < 0.10 else ("WATCH" if v < 0.25 else "ALERT")
    print(f"{f:<12}{v:>8.4f}  {status}")

shifted_scores = plain.predict_proba(drifted)[:, 1]
print(f"\nmean risk score  baseline window {plain.predict_proba(stable)[:, 1].mean():.4f}"
      f"  ->  drifted window {shifted_scores.mean():.4f}")
print(f"review-band share  {(( shifted_scores >= 0.15) & (shifted_scores < 0.55)).mean():.1%}")

**Observe:** the drift report and the downstream effect:

```
feature          PSI  status
LIMIT_BAL     0.3874  ALERT
AGE           0.0091  OK
PAY_0         0.2681  ALERT
BILL_AMT1     0.0118  OK
PAY_AMT1      0.0136  OK

mean risk score  baseline window 0.2208  ->  drifted window 0.3402
review-band share  49.8%
```

**Infer:** the two features that were deliberately shifted alert; the three untouched
features stay at their noise floor. That contrast is the point — PSI localizes the drift to
specific columns, so the on-call engineer's first question ("did the world change, or did an
upstream job change?") has a starting answer. Drift on `LIMIT_BAL` and `PAY_0` *together*
is a coherent economic story; drift on one feature alone, with the others perfectly stable,
is far more often a broken upstream pipeline than a changed population.

The operational consequence is in the last two lines. Mean risk moved 0.221 → 0.340 and the
review band swelled to **49.8%** of traffic — the operations team is now drowning, and they
will notice before any accuracy metric can. Input drift is not an abstract statistic; it
propagates within hours into review queue depth, and that queue is often the real alarm.

Bear in mind the caveat from the top: this simulated drift is *gradual and economy-shaped*.
Adversarial fraud drift arrives as a step change over hours as an attacker finds the
boundary, which means a daily PSI job is too slow and you need windowed monitoring at the
hour scale plus alerting on the score distribution itself, not only on inputs.

## When the monitoring is the problem

Two failure modes that will cost you credibility faster than a mediocre model.

### The alert that was a schema change

**Symptom:** PSI on one feature spikes to 3.5 overnight while every other feature is
perfectly stable and the business reports nothing unusual.

**Observe:** the feature's `min`, `max`, and `dtype` in the current window against the
baseline. A PSI above ~1.0 on a single column is almost never a population shift —
populations do not move that far that fast.
**Infer:** you are looking at a broken contract, not a changed world. The usual causes:
an upstream job started sending `LIMIT_BAL` in dollars instead of thousands (values ×1000,
PSI enormous); a NULL began arriving as `0` after a join changed; a categorical was recoded
so `EDUCATION=4` now means what `5` used to. The response is completely different from a
genuine-drift response — you fix the pipeline, you do **not** retrain, and retraining on
corrupted inputs bakes the corruption into the model. Distinguish them by magnitude and by
whether the shift is a *translation of the whole distribution* (usually a units or encoding
bug) versus a *change in shape* (usually real). Asserting min/max ranges and dtypes at
ingestion catches this class before PSI ever sees it.

### The threshold that decayed quietly

**Symptom:** no drift alerts, stable PSI everywhere, and the review queue slowly growing
over eight weeks.

**Observe:** the *share of traffic in each decision band* over time — plot approve /
review / decline percentages weekly alongside PSI.
**Infer:** PSI watches inputs one feature at a time and is blind to shifts in the
*relationship* between features and outcome, and to gradual drift in the model's own
calibration. The score distribution can migrate upward while every marginal input
distribution stays put. Since your thresholds were fitted to a specific calibration
(Step 6), the same 0.15 cut now means something different, and your effective risk policy
has changed without anyone deciding to change it. Monitoring band shares is the cheap
detector; the fix is to re-derive thresholds on recent labelled data whenever labels mature,
on a schedule, independently of retraining. Recalibrating a threshold is hours of work;
noticing after two quarters is not.

## What this analogy leaves out

Worth stating explicitly before you carry this design to actual fraud:

* **Adversarial adaptation.** Credit-default drift is passive; fraud drift is an opponent
  probing for your boundary. Assume any published threshold is discoverable through
  repeated trials, and expect step-change drift over hours rather than months.
* **Extreme imbalance.** At 0.5% positives, PR-AUC is far noisier, a single test-set batch
  is not enough to fit a threshold on, and resampling (dismissed in Step 4 at 22%) becomes
  a real consideration.
* **Behavioural features.** The strongest fraud features are velocity and relational —
  transactions per hour, geo-distance from the last one, device seen before — which means a
  low-latency **feature store** lookup sits inside the budget measured in Step 7, and it is
  usually slower than the model.
* **Label lag with a twist.** Fraud labels arrive not just late but *biased*: you only
  learn the outcome of transactions you approved. Everything you declined has no label
  ever, so your training data is a censored sample of your own past decisions.

## What to try next

* Re-run Step 6's cost sweep with `COST_FP = 200` (a customer-facing decline rather than a
  back-office review). Watch the optimal threshold rise and the model finally beat the
  review-everyone baseline — the clearest demonstration that a threshold is a business
  parameter, not a model parameter.
* Replace the hand-rolled PSI in Step 8 with **Session 5**'s Evidently AI drift reports, and
  wire the alert into **Session 17**'s automated retraining trigger so a sustained ALERT
  starts a retrain instead of paging someone.
* Put **Session 22**'s SHAP explanations behind the `review` decision band. An analyst
  handed "risk 0.42, review" can do nothing with it; "risk 0.42, driven by PAY_0=2 and a
  falling payment ratio" is an actionable case file — and in regulated lending an
  explanation is often legally required.
* Serve `score_service.py` through **Session 23**'s BentoML packaging and re-measure the
  p99 from Step 7 *over HTTP*. The network and serialization hop usually dominates the
  0.5 ms of model time, and knowing the split tells you which one to optimize.
* Gate retrained versions of this model with **Session 24**'s quality gate, but make the
  gated metric **expected cost** rather than accuracy — that is the only threshold-aware
  gate that means anything for a cost-sensitive decision system.